# Day 4 — DistilBERT Sentiment Classifier

Fine-tuning a pre-trained transformer model and beating the 79.2% baseline.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

print("PyTorch version:", torch.__version__)
print("Device:", "MPS (Apple GPU)" if torch.backends.mps.is_available() else "CPU")

df = pd.read_csv("train.tsv", delimiter='\t', header=None, names=['text', 'label'])
print("Data loaded:", df.shape)

PyTorch version: 2.12.0
Device: MPS (Apple GPU)
Data loaded: (6920, 2)


In [2]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Test it on a sample sentence
sample = "This movie was absolutely fantastic!"
tokens = tokenizer(sample, return_tensors="pt")
print("Input IDs:", tokens['input_ids'])
print("Tokens:", tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Input IDs: tensor([[  101,  2023,  3185,  2001,  7078, 10392,   999,   102]])
Tokens: ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'fantastic', '!', '[SEP]']


In [3]:
from torch.utils.data import Dataset, DataLoader

# Tokenise all reviews
encodings = tokenizer(
    list(df['text']),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
)

print("Input IDs shape:", encodings['input_ids'].shape)
print("Attention mask shape:", encodings['attention_mask'].shape)


Input IDs shape: torch.Size([6920, 64])
Attention mask shape: torch.Size([6920, 64])


In [4]:
from torch.utils.data import Dataset, DataLoader, random_split

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

dataset = ReviewDataset(encodings, list(df['label']))

# 80/20 split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 173
Test batches: 44


In [5]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

# Load model
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", 
    num_labels=2
)
model = model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

print("Model loaded on:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "million")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on: mps
Parameters: 66.95501 million


In [6]:
from torch.nn.functional import cross_entropy

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

print("Starting training...")
for epoch in range(3):
    loss, acc = train_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch+1}/3 — Loss: {loss:.4f} | Train Accuracy: {acc:.4f}")

Starting training...
Epoch 1/3 — Loss: 0.4132 | Train Accuracy: 0.8112
Epoch 2/3 — Loss: 0.1939 | Train Accuracy: 0.9290
Epoch 3/3 — Loss: 0.1000 | Train Accuracy: 0.9691


In [7]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, all_labels

preds, labels = evaluate(model, test_loader, device)
acc = accuracy_score(labels, preds)
print(f"Test Accuracy: {acc:.4f}")
print(f"Baseline (Logistic Regression): 0.7919")
print(f"Improvement: +{acc - 0.7919:.4f}")
print("\n--- DistilBERT Report ---")
print(classification_report(labels, preds, target_names=['Negative', 'Positive']))

Test Accuracy: 0.8895
Baseline (Logistic Regression): 0.7919
Improvement: +0.0976

--- DistilBERT Report ---
              precision    recall  f1-score   support

    Negative       0.93      0.85      0.88       694
    Positive       0.86      0.93      0.89       690

    accuracy                           0.89      1384
   macro avg       0.89      0.89      0.89      1384
weighted avg       0.89      0.89      0.89      1384



## DistilBERT Results

| Model | Accuracy | F1 |
|-------|----------|----|
| Logistic Regression (baseline) | 79.2% | 0.79 |
| Naive Bayes | 79.8% | 0.79 |
| **DistilBERT (fine-tuned)** | **88.9%** | **0.89** |

**Improvement over baseline: +9.7%**

- Precision 93% on negative, 86% on positive
- Recall 85% on negative, 93% on positive
- 3 epochs, batch size 32, lr=2e-5, Apple MPS GPU
- Training time: ~3 minutes

In [8]:
import os
os.makedirs("model", exist_ok=True)
model.save_pretrained("model/distilbert-sentiment")
tokenizer.save_pretrained("model/distilbert-sentiment")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!
